# 🚀 DataMate AI Zalo Challenge 2026 - Colab Pipeline Runner

Notebook này chạy toàn bộ pipeline chuẩn hóa và suy luận tự động của DataMate trên Google Colab,
đã tích hợp đầy đủ các bản vá lỗi (sửa lỗi auto repair, bỉm tã, nhận diện ngành kinh doanh, từ điển tiếng lóng, v.v.).

> **Lưu ý:** Pipeline là deterministic rule-based engine, chạy siêu tốc trên CPU (~40s cho 1.000 câu hỏi), không yêu cầu GPU.

## Bước 1: Clone Repository & Cài đặt Thư viện

In [ ]:
# Clone nhánh thang-bu-win-vinnie đã chứa toàn bộ bản vá lỗi mới nhất
import os
WORKDIR = '/content/datamate'
if not os.path.exists(WORKDIR):
    !git clone -b thang-bu-win-vinnie https://github.com/vohoangloc-32/datamate.git {WORKDIR}
else:
    !cd {WORKDIR} && git pull origin thang-bu-win-vinnie

%cd {WORKDIR}
!pip install -q openpyxl xlrd tabulate pyyaml pytest pandas
!git log -1 --oneline

## Bước 2: Chuẩn bị Thư mục dữ liệu `public_test`

Vì dữ liệu cuộc thi không được commit lên GitHub, hãy nạp thư mục `public_test` vào `/content/datamate/public_test` bằng 1 trong 2 cách:

In [ ]:
# Cách A: Upload file public_test.zip trực tiếp từ máy lên Colab
from google.colab import files
import os

if not os.path.exists('/content/datamate/public_test'):
    print('Vui lòng chọn file public_test.zip để upload:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            !unzip -q {fn} -d /content/datamate/
            print('Đã giải nén public_test thành công!')
else:
    print('Thư mục public_test đã sẵn sàng!')

> **Hoặc Cách B:** Nếu bạn lưu `public_test` trên Google Drive, mount Drive và copy sang:
```python
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/public_test' /content/datamate/
```

## Bước 3: Chạy Pipeline & Tạo `submission.jsonl`

In [ ]:
%cd /content/datamate
!python scripts/run_baseline.py --split public_test --output submission.jsonl --trace artifacts/traces_public_test.jsonl

## Bước 4: Kiểm tra tính hợp lệ & Thống kê kết quả

In [ ]:
import json
from collections import Counter
import shutil

with open('submission.jsonl', 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

print(f'Tổng số dòng: {len(records)}/1000')
counts = Counter(r['type'] for r in records)
print('Phân phối kiểu trả lời:')
for t, c in counts.most_common():
    print(f'  {t:15s}: {c}')

nulls = sum(1 for r in records if r['answer'] is None)
print(f'Số câu trả lời null (unanswerable + abstain): {nulls}')

# Tạo thêm bản sao submission.json nếu người dùng cần đuôi .json
shutil.copy('submission.jsonl', 'submission.json')
print('Đã tạo cả submission.jsonl và submission.json!')

## Bước 5: 1-Click Tải file Submission về Máy

In [ ]:
from google.colab import files
import os

if os.path.exists('submission.jsonl'):
    print('Đang tải submission.jsonl về máy tính...')
    files.download('submission.jsonl')
    # Hoặc tải file submission.json:
    # files.download('submission.json')
else:
    print('Lỗi: Không tìm thấy submission.jsonl')